# ILORA Data Pipeline
## Notebook 1 of 2 - From Raw Database to AI-Ready Corpus

---

### What is ILORA?

ILORA (Indian Alien Flora Information) is India's first comprehensive
database of alien vascular plant species, curated with information on
1,747 species across 16 relational tables covering taxonomy, geographic distribution, habitat, invasion status, introduction pathways, native range, economic uses, ecoregions, and occurrence records.
Despite its richness, ILORA was not designed for natural language
querying.

### The Problem

General-purpose AI tools like ChatGPT are increasingly used to query
ecological information. However, they are not grounded in verified,
India-specific data and frequently hallucinate - producing outputs that
appear coherent but are factually incorrect. This is particularly
dangerous in ecological decision-making, where incorrect species
identification or risk assessment can lead to harmful interventions.

At the same time, ILORA's data remains locked in a format that cannot
be directly queried using natural language or semantic search.

### What This Notebook Does

This notebook builds the data foundation for a domain-specific
ecological decision-support system. It:

1. Loads all 16 ILORA tables from Google Drive
2. Explores data completeness across tables
3. Converts binary matrices and coded fields into readable text
4. Builds one structured text chunk per species
5. Saves 1,747 species chunks as a JSON corpus for downstream AI use

### Output

`ilora_chunks.json` - 1,747 species text chunks, ready for
semantic embedding, FAISS indexing, and Retrieval-Augmented Generation.

### Tech Stack

| Component | Tool |
|---|---|
| Data processing | pandas, json |
| Progress tracking | tqdm |
| Storage | Google Drive |
| License | All open source (MIT / Apache 2.0) |

## Section 2 - Setup & Data Loading

This section mounts Google Drive, installs required libraries,
and loads all 16 ILORA tables into memory.

Each table is stored as a separate CSV file exported from the
ILORA database.

In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
import os
from google.colab import drive


In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
base_path = '/content/drive/MyDrive/ILORA_PoC/data/'

table_names = [
    "ILORA_1_SpCategorization",
    "ILORA_2_GeneralInformation",
    "ILORA_3_NativeRange",
    "ILORA_4_Introduction",
    "ILORA_5_EconomicUses",
    "ILORA_6_MarketDynamics",
    "ILORA_7_Habitat",
    "ILORA_8_NaturalizedRange",
    "ILORA_9_Occurrence",
    "ILORA_10_Distribution",
    "ILORA_11_LULC",
    "ILORA_12_Anthrome",
    "ILORA_13_Ecoregions",
    "ILORA_14_Climate",
    "ILORA_15_Summary",
    "ILORA_16_CultivatedSpecies"
]
data = {}
for table in table_names:
    data[table] = pd.read_csv(f"{base_path}{table}.csv")
    print(f"{table}: {data[table].shape}")

ILORA_1_SpCategorization: (1747, 10)
ILORA_2_GeneralInformation: (1747, 8)
ILORA_3_NativeRange: (7942, 5)
ILORA_4_Introduction: (1747, 8)
ILORA_5_EconomicUses: (1747, 54)
ILORA_6_MarketDynamics: (1747, 12)
ILORA_7_Habitat: (1747, 41)
ILORA_8_NaturalizedRange: (73786, 6)
ILORA_9_Occurrence: (40716, 5)
ILORA_10_Distribution: (1747, 37)
ILORA_11_LULC: (1747, 39)
ILORA_12_Anthrome: (1747, 21)
ILORA_13_Ecoregions: (1747, 45)
ILORA_14_Climate: (1747, 27)
ILORA_15_Summary: (1747, 16)
ILORA_16_CultivatedSpecies: (756, 6)


In [4]:
# Drop duplicate fuels column
data['ILORA_5_EconomicUses'] = data['ILORA_5_EconomicUses'].drop(
    columns=['0700_Fuels.1'])

In [5]:
print("Setup complete.")
print(f"Total tables loaded: {len(data)}")
print(f"Total unique species: {len(data['ILORA_2_GeneralInformation'])}")

Setup complete.
Total tables loaded: 16
Total unique species: 1747


### Note on Table Sizes

Three tables have more than 1,747 rows because they store
multiple records per species:
- `ILORA_3_NativeRange` - multiple native regions per species
- `ILORA_8_NaturalizedRange` - multiple naturalized locations
- `ILORA_9_Occurrence` - multiple GPS observation points

One table has fewer rows:
- `ILORA_16_CultivatedSpecies` - only 756 species are cultivated

## Section 3 - Data Exploration

Before building the pipeline we need to understand what the raw
data looks like and where the gaps are.

ILORA stores most information as binary matrices - columns represent
attributes, values are 1.0 (present) or NaN (absent). This section
shows why transformation into readable text is necessary.

### 3.1 - Raw Data Sample

Below is the raw data for one species across three tables.
Notice how the information is encoded.

In [6]:
test_species = "Acacia auriculiformis Benth."

print("=== General Information ===")
display(data['ILORA_2_GeneralInformation'][
    data['ILORA_2_GeneralInformation']['Acc_Species_Name'] == test_species
])

=== General Information ===


,acc_species_id,Acc_Species_Name,Common.Name,Vernacular.Name,Growth.Habit,Duration,Group,Source
2,5,Acacia auriculiformis Benth.,Northern Black Wattle,"Kalsiamthing (lus), Thumma (tel)",Shrub;Tree,Perennial,Dicot,TRY;NRCS


In [7]:
print("=== Distribution (first 10 columns) ===")
dist = data['ILORA_10_Distribution'][
    data['ILORA_10_Distribution']['Acc_Species_Name'] == test_species
]
display(dist.iloc[:, :10])

=== Distribution (first 10 columns) ===


,acc_species_id,Acc_Species_Name,Kerala,Maharashtra,Madhya Pradesh,Gujarat,Rajasthan,West Bengal,Tamil Nadu,Karnataka
2,5,Acacia auriculiformis Benth.,NaN,1.0,NaN,1.0,NaN,1.0,1.0,1.0


In [8]:
print("=== Economic Uses (first 10 columns) ===")
econ = data['ILORA_5_EconomicUses'][
    data['ILORA_5_EconomicUses']['Acc_Species_Name'] == test_species
]

display(econ.iloc[:, :10])

=== Economic Uses (first 10 columns) ===


,acc_species_id,Acc_Species_Name,0100_Food,0200_Food_Additives,0200_Spices/Herbs,0300_Forage,0300_Fodder,0400_Honey,0500_InvFood,0600_Timber
2,5,Acacia auriculiformis Benth.,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0


### Observation

The Distribution and Economic Uses tables use binary encoding
`1.0` means present, `NaN` means absent. Column names are the
actual state names or use categories. These need to be converted
into readable text before they can be used for AI querying.

### 3.2 - Data Completeness

The Summary table tracks which tables have data for each species.
Many species are missing data in several tables - these gaps are
handled by fallback text in the chunk builder.

In [9]:
summary = data['ILORA_15_Summary']
null_counts = summary.isnull().sum()
print("Missing data per table:")
print(null_counts)

Missing data per table:
acc_species_id            0
Acc_Species_Name          0
Sp.Categorisation         0
General.Information       0
Native.Range            303
Introduction            709
Economic.Uses           591
Market.Dynamics        1305
Habitat                1288
Naturalized.Range       403
Occurrence              858
Distribution            859
LULC                    867
Anthromes               861
Ecoregions              859
Climate                 856
dtype: int64


### Key Insight

The data is rich but fragmented and binary-encoded. Direct natural
language querying is impossible without transformation.

The chunk builder in Section 5 converts this raw data into readable
text enabling semantic search and AI-powered querying.

## Section 4 - Helper Functions

Four reusable functions handle the core data extraction logic.
Each function is designed to work across any of the 16 ILORA tables.

These functions are the building blocks of the chunk builder in Section 5.

### Function 1 - get_species_details()

Retrieves all rows for a given species from a specified table.
Returns an empty DataFrame if the species is not found.

In [11]:
# Function 1
def get_species_details(data, table, species_name):
    """Get all rows for a given species from the specified table."""
    df = data[table]
    return df[df["Acc_Species_Name"] == species_name]

### Function 2 - get_col_with_1()

For binary matrix tables, returns a list of column names where
the value is 1.0 for a given species. Metadata columns are excluded.

Used for: Distribution, Economic Uses, Ecoregions, Habitat, LULC, Anthrome

In [12]:
# Function 2
def get_col_with_1(data, table, species_name):
    """Return list of columns with value 1.0 for the given species."""
    df = get_species_details(data, table, species_name)
    if df.empty:
        return []
    row = df.iloc[0]
    return [
        col for col in row.index
        if row[col] == 1.0
        and col not in ['id', 'acc_species_id', 'Acc_Species_Name',
                        'Source', 'SOURCE', 'index']
    ]

### Function 3 - get_joined_values()

For tables with multiple rows per species (like NativeRange),
retrieves all values from a specified column and joins them
into a single comma-separated string.

Used for: NativeRange, NaturalizedRange

In [13]:
# Function 3
def get_joined_values(data, table, species_name, column_to_join, sep=', '):
    """Get all values in a column for a given species joined as one string."""
    df = data[table]
    filtered = df[df["Acc_Species_Name"] == species_name]
    values = filtered[column_to_join].dropna()
    return sep.join(values.astype(str).tolist())

### Function 4 - count_occurrence_record()

Counts how many GPS observation records exist for a given species
in the Occurrence table.

In [14]:
# Function 4
def count_occurrence_record(species_name):
    """Count the number of occurrence records for a given species."""
    df = data['ILORA_9_Occurrence']
    return df[df['Acc_Species_Name'] == species_name].shape[0]


### Function 5 - clean_economic_use(col_name)

Cleans column names by removing the first prefix (before the first underscore) and replacing remaining underscores with spaces.

Used for: standardizing economic use and similar structured column fields

### Function 6 - fallback_clean(col_name)

Fallback column cleaner for irregular naming formats.
Removes prefix before first underscore, replaces underscores with spaces, and formats / for readability.

Used for: inconsistent or legacy column names

In [15]:
# ---------------------------------------------------------
# Helper Functions: Column Name Cleaning Utilities
# ---------------------------------------------------------

def clean_economic_use(col_name):
    """
    Cleans column names by removing the first prefix
    (before the first underscore) and replacing remaining
    underscores with spaces.
    """
    if not isinstance(col_name, str):
        return col_name

    parts = col_name.split('_', 1)
    cleaned = parts[1] if len(parts) > 1 else parts[0]
    cleaned = cleaned.replace('_', ' ').strip()

    return cleaned


def fallback_clean(col_name):
    """
    Fallback cleaner for column names.
    Removes first prefix, replaces underscores with spaces,
    and formats '/' with spacing for readability.
    """
    if not isinstance(col_name, str):
        return col_name

    parts = col_name.split('_', 1)
    cleaned = parts[1] if len(parts) > 1 else parts[0]
    return cleaned.replace('_', ' ').replace('/', ' / ').strip()

### 4.1 - Function Tests

Testing all four functions on a single species to verify correctness.

In [16]:
test_species = "Acacia auriculiformis Benth."

print("Test 1 - get_species_details()")
print(get_species_details(data, 'ILORA_2_GeneralInformation', test_species))
print()

print("Test 2 - get_col_with_1() on Distribution")
print(get_col_with_1(data, 'ILORA_10_Distribution', test_species))
print()

print("Test 3 - get_joined_values() on NativeRange")
print(get_joined_values(data, 'ILORA_3_NativeRange',
                         test_species, 'TDWG_Level2_Names'))
print()

print("Test 4 - count_occurrence_record()")
print(count_occurrence_record(test_species))

Test 1 - get_species_details()
   acc_species_id              Acc_Species_Name            Common.Name  \
2               5  Acacia auriculiformis Benth.  Northern Black Wattle   

                    Vernacular.Name Growth.Habit   Duration  Group    Source  
2  Kalsiamthing (lus), Thumma (tel)   Shrub;Tree  Perennial  Dicot  TRY;NRCS  

Test 2 - get_col_with_1() on Distribution
['Maharashtra', 'Gujarat', 'West Bengal', 'Tamil Nadu', 'Karnataka', 'Goa', 'Chhattisgarh', 'Tripura', 'Assam', 'Uttar Pradesh', 'Jammu and Kashmir']

Test 3 - get_joined_values() on NativeRange
Papuasia, Australia

Test 4 - count_occurrence_record()
26


## Section 5 - Chunk Builder

The chunk builder combines data from multiple ILORA tables into
one readable text paragraph per species.

Two transformations are applied before building:
1. **Invasion status codes** -> readable words (e.g. "In" → "Invasive")
2. **Economic use column codes** -> clean readable labels (e.g. "0600_Timber" → "Timber")

Missing data is handled with explicit fallback text so every
species produces a valid chunk.

### 5.1 - Lookup Dictionaries

In [17]:
# Invasion status code to readable label
invasion_status_map = {
    'In': 'Invasive',
    'Nt': 'Naturalized',
    'CA': 'Casual Alien',
    'CG': 'Cryptogenic',
    'N':  'Native'
}

# Economic uses column code to readable label
economic_uses_map = {
    '0100_Food': 'Food',
    '0200_Food_Additives': 'Food Additives',
    '0200_Spices/Herbs': 'Spices / Herbs',
    '0300_Forage': 'Forage',
    '0300_Fodder': 'Fodder',
    '0400_Honey': 'Honey',
    '0500_InvFood': 'Invertebrate Food',
    '0600_Timber': 'Timber',
    '0600_EssentialOil': 'Essential Oil',
    '0600_Dye': 'Dye',
    '0600_Gum': 'Gum',
    '0600_Fibre': 'Fibre',
    '0600_Lipid': 'Lipid',
    '0600_Wax': 'Wax',
    '0600_CarvedMaterial': 'Carved Material',
    '0600_BarkProducts': 'Bark Products',
    '0600_Cosmetic': 'Cosmetic',
    '0600_Pesticide/Fertilizer': 'Pesticide / Fertilizer',
    '0600_Oil': 'Oil',
    '0600_Chemical': 'Chemical',
    '0600_Latex/Rubber': 'Latex / Rubber',
    '0600_Beads': 'Beads',
    '0600_Basket': 'Basket',
    '0600_Cane': 'Cane',
    '0600_WrappingPaper': 'Wrapping Paper',
    '0600_Alcohol': 'Alcohol',
    '0700_Fuels': 'Fuelwood',
    '0800_SocialUse': 'Social Use',
    '0900_VertPoison': 'Vertebrate Poison',
    '1000_InvertPoison': 'Invertebrate Poison',
    '1100_Medicine': 'Medicine',
    '1100_TraditionalMedicine': 'Traditional Medicine',
    '1200_Agrooforestry': 'Agroforestry',
    '1200_Shade': 'Shade',
    '1200_Barrier': 'Barrier',
    '1200_SoilImprover': 'Soil Improver',
    '1200_ErosionControl': 'Erosion Control',
    '1200_Revegetator': 'Revegetator',
    '1200_Phytoremedy': 'Phytoremedy',
    '1200_LandscapeImprover': 'Landscape Improver',
    '1200_GreenManure': 'Green Manure',
    '1200_Mulches': 'Mulches',
    '1200_LandReclamation': 'Land Reclamation',
    '1200_SoilConservation': 'Soil Conservation',
    '1200_Amenity': 'Amenity',
    '1200_Lawn/Turf': 'Lawn / Turf',
    '1200_PollutionControl': 'Pollution Control',
    '1200_OrnamentalUse': 'Ornamental Use',
    '1300_GeneSource': 'Gene Source',
    '1300_ResearchModel': 'Research Model'
}

print(f"Invasion status codes: {len(invasion_status_map)}")
print(f"Economic use labels: {len(economic_uses_map)}")

Invasion status codes: 5
Economic use labels: 50


### 5.2 - build_ilora_chunk()

The master function that assembles one text chunk per species
by pulling from all relevant tables using the helper functions.

In [20]:
def build_ilora_chunk(data, species_name):
    """
    Builds a clean text chunk for one species by combining
    multiple ILORA tables into one readable paragraph.
    """
    parts = []

    # Species name - always first
    parts.append(f"Species: {species_name}")

    # --- Invasion Status ---
    sp_cat = get_species_details(data, 'ILORA_1_SpCategorization', species_name)
    if not sp_cat.empty:
        raw_status = sp_cat.iloc[0].get('Invasion_Status', '')
        status = invasion_status_map.get(raw_status, raw_status)
        parts.append(f"\nInvasion Status: {status}")

    # --- General Information ---
    gen = get_species_details(data, 'ILORA_2_GeneralInformation', species_name)
    if not gen.empty:
        row = gen.iloc[0]
        info_parts = []
        for col in ['Common.Name', 'Vernacular.Name',
                    'Growth.Habit', 'Duration', 'Group']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                info_parts.append(f"{col}: {val}")
        if info_parts:
            parts.append("\nGeneral Information:\n" +
                        "\n".join(info_parts))

    # --- Introduction Pathway ---
    intro = get_species_details(data, 'ILORA_4_Introduction', species_name)
    if not intro.empty:
        intro_text = intro.iloc[0].get(
            'Introduction_pathways_reported', '')
        if pd.notna(intro_text) and str(intro_text).strip():
            parts.append(f"\nIntroduction:\n{intro_text}")

    # --- Native Range ---
    native = get_joined_values(
        data, 'ILORA_3_NativeRange',
        species_name, 'TDWG_Level2_Names'
    )
    if native.strip():
        parts.append(f"\nNative Range: {native}")

    # --- Distribution in India ---
    dist_cols = get_col_with_1(
        data, 'ILORA_10_Distribution', species_name)
    if dist_cols:
        parts.append(f"\nDistribution: {', '.join(dist_cols)}")
    else:
        parts.append("\nDistribution: Not available")

    # --- Economic Uses ---
    econ_cols = get_col_with_1(
        data, 'ILORA_5_EconomicUses', species_name)
    if econ_cols:
        clean_uses = [
            economic_uses_map.get(col, col.split('_', 1)[-1]
                                  .replace('_', ' '))
            for col in econ_cols
        ]
        clean_uses = list(dict.fromkeys(clean_uses))
        parts.append(f"\nEconomic Uses: {', '.join(clean_uses)}")
    else:
        parts.append("\nEconomic Uses: Not available")

    # --- Ecoregions ---
    eco_cols = get_col_with_1(
        data, 'ILORA_13_Ecoregions', species_name)
    if eco_cols:
        parts.append(f"\nEcoregions: {', '.join(eco_cols)}")

    # --- Occurrence Records ---
    occ_count = count_occurrence_record(species_name)
    parts.append(f"\nOccurrence Records: {occ_count}")

    return "\n".join(parts)

### 5.3 - Chunk Test

Testing the chunk builder on three species with different
levels of data completeness.

In [21]:
test_species_list = [
    "Acacia auriculiformis Benth.",       # rich data
    "Abelmoschus manihot (L.) Medik.",    # medium data
    "Abutilon auritum (Wall. ex Link) Sweet"  # sparse data
]

for sp in test_species_list:
    print("=" * 70)
    print(build_ilora_chunk(data, sp))
    print()

Species: Acacia auriculiformis Benth.

Invasion Status: Invasive

General Information:
Common.Name: Northern Black Wattle
Vernacular.Name: Kalsiamthing (lus), Thumma (tel)
Growth.Habit: Shrub;Tree
Duration: Perennial
Group: Dicot

Introduction:
Wood;Ornamental

Native Range: Papuasia, Australia

Distribution: Maharashtra, Gujarat, West Bengal, Tamil Nadu, Karnataka, Goa, Chhattisgarh, Tripura, Assam, Uttar Pradesh, Jammu and Kashmir

Economic Uses: Honey, Invertebrate Food, Timber, Dye, Gum, Fibre, Carved Material, Bark Products, Fuelwood, Traditional Medicine, Agroforestry, Shade, Soil Improver, Erosion Control, Revegetator, Mulches, Ornamental Use

Ecoregions: Malabar Coast Moist Forests, Deccan Thorn Scrub Forests, South Western Ghats Moist Deciduous Forests, South Deccan Plateau Dry Deciduous Forests, East Deccan Dry-Evergreen Forests, Narmada Valley Dry Deciduous Forests, Eastern Highlands Moist Deciduous Forests, Northwestern Thorn Scrub Forests, Lower Gangetic Plains Moist Decid

## Section 6 - Generate & Save Corpus

All 1,747 species are processed in sequence using the chunk builder.
Each chunk is stored as a dictionary with two keys:
- `species` - the accepted species name
- `text` - the full readable text chunk

The corpus is saved as a JSON file to Google Drive for use in
Notebook 2 (RAG System).

### 6.1 - Generate All Chunks

In [22]:
import json
import os
from tqdm import tqdm

species_list = data['ILORA_2_GeneralInformation'][
    'Acc_Species_Name'].dropna().unique()

all_chunks = []

for sp in tqdm(species_list, desc="Building ILORA chunks"):
    try:
        chunk_text = build_ilora_chunk(data, sp)
        all_chunks.append({
            "species": sp,
            "text": chunk_text
        })
    except Exception as e:
        print(f"Error processing {sp}: {e}")

print(f"\nTotal chunks generated: {len(all_chunks)}")

Building ILORA chunks: 100%|██████████| 1747/1747 [00:12<00:00, 137.87it/s]


Total chunks generated: 1747


### 6.2 - Save to Google Drive

In [23]:
output_dir = "/content/drive/MyDrive/ILORA_PoC/chunks/"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "ilora_chunks.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/ILORA_PoC/chunks/ilora_chunks.json


### 6.3 - Verify Save

In [24]:
with open(output_path, 'r', encoding="utf-8") as f:
    verify = json.load(f)

print(f"Chunks loaded back: {len(verify)}")
print(f"First species: {verify[0]['species']}")
print(f"Last species: {verify[-1]['species']}")
print()
print("Sample chunk:")
print("=" * 60)
print(verify[2]['text'])

Chunks loaded back: 1747
First species: Abelmoschus manihot (L.) Medik.
Last species: Zoysia japonica Steud.

Sample chunk:
Species: Acacia auriculiformis Benth.

Invasion Status: Invasive

General Information:
Common.Name: Northern Black Wattle
Vernacular.Name: Kalsiamthing (lus), Thumma (tel)
Growth.Habit: Shrub;Tree
Duration: Perennial
Group: Dicot

Introduction:
Wood;Ornamental

Native Range: Papuasia, Australia

Distribution: Maharashtra, Gujarat, West Bengal, Tamil Nadu, Karnataka, Goa, Chhattisgarh, Tripura, Assam, Uttar Pradesh, Jammu and Kashmir

Economic Uses: Honey, Invertebrate Food, Timber, Dye, Gum, Fibre, Carved Material, Bark Products, Fuelwood, Traditional Medicine, Agroforestry, Shade, Soil Improver, Erosion Control, Revegetator, Mulches, Ornamental Use

Ecoregions: Malabar Coast Moist Forests, Deccan Thorn Scrub Forests, South Western Ghats Moist Deciduous Forests, South Deccan Plateau Dry Deciduous Forests, East Deccan Dry-Evergreen Forests, Narmada Valley Dry Decid

## Section 7 - Corpus Statistics

Final verification and analysis of the generated corpus before
passing it to the RAG system in Notebook 2.

### 7.1 - Basic Statistics

In [25]:
chunk_lengths = [len(c['text']) for c in all_chunks]

avg_length = sum(chunk_lengths) // len(chunk_lengths)
min_length = min(chunk_lengths)
max_length = max(chunk_lengths)

min_species = all_chunks[chunk_lengths.index(min_length)]['species']
max_species = all_chunks[chunk_lengths.index(max_length)]['species']

print(f"Total chunks:          {len(all_chunks)}")
print(f"Average chunk length:  {avg_length} characters")
print(f"Shortest chunk:        {min_length} characters - {min_species}")
print(f"Longest chunk:         {max_length} characters - {max_species}")

Total chunks:          1747
Average chunk length:  583 characters
Shortest chunk:        211 characters - Fagonia bruguieri DC.
Longest chunk:         2563 characters - Colocasia esculenta (L.) Schott


### 7.2 - Shortest Chunk

The shortest chunk represents a species with minimal data in ILORA.
Even sparse species produce a valid chunk with fallback text.

In [27]:
shortest_idx = chunk_lengths.index(min_length)
print("Shortest chunk:")
print("=" * 60)
print(all_chunks[shortest_idx]['text'])

Shortest chunk:
Species: Fagonia bruguieri DC.

Invasion Status: Native

General Information:
Growth.Habit: Herb
Duration: Perennial
Group: Dicot

Distribution: Not available

Economic Uses: Not available

Occurrence Records: 0


### 7.3 - Longest Chunk

The longest chunk represents a species with data across
almost all ILORA tables.

In [28]:
longest_idx = chunk_lengths.index(max_length)
print("Longest chunk:")
print("=" * 60)
print(all_chunks[longest_idx]['text'])

Longest chunk:
Species: Colocasia esculenta (L.) Schott

Invasion Status: Native

General Information:
Common.Name: Taro
Vernacular.Name: Bon kachu, pani kachu (asm); banakochu, jongli kochu (ben); arui, kanda (Bhojpuri); alavi (guj); arabi, aruwi, banda, ghuyan, kachalu, kachchu, kala kachchu, kechuk, manak kanchu, nalika, nalita, nari patra, narich, pechu, van kachu, vishva rochan, vitanda (hin); kesavu,kesa, kesu (kan); terem, venti (knn); chemp, manam, tal (mal); Pan (mni); alem, alu (mar); dawl (mar); aruvee, gaawaa, karkalo, kuchuro, pindaalu (npi); kachu, pechu, saru (ori); gagli, gawian, kachalu (pan); dalasarini, kachu, kachvi, kalakachu, kemuka, nadicha, nadipattra, nalita, pechu, shakata, trutibija, vanakachu, vishvarochana, vitanda (san); kachalu (Sindhi); chempu, nir-c-cempu, peculam (tam); chama, chema (tel); aruwi, ghuyan, kachalu, kachchu, kechuk, nalika, nalita, pechu, narich, vitanda (urd)
Growth.Habit: Herb
Duration: Perennial
Group: Monocot

Introduction:
Ornamental